# 01 — Análisis Exploratorio de Datos (EDA)

**TFM RunnAing** | Dataset: FitRec (Ni et al., 2019)

Objetivos:
1. Distribución de variables (FC, velocidad, altitud, duración) — histogramas + estadísticos
2. Detección de outliers mediante IQR y z-score — boxplots
3. Análisis de correlaciones — heatmap
4. Sesgos demográficos — sexo, sesiones por usuario
5. Span temporal por usuario — cohorte ACWR (≥ 28 días) y sesgo de cobertura

Outputs guardados en `reports/eda/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats

from src.data_loader import auto_detect_and_load, filter_running_sessions

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

REPORTS = Path('../reports/eda')
REPORTS.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK — outputs en reports/eda/ y reports/figures/')

## 1. Carga del dataset

> **Prerrequisito**: colocar el archivo FitRec en `data/raw/fitrec.jsonl` (o `.csv`).
> Ver README.md para instrucciones de descarga.

In [ ]:
df_raw = auto_detect_and_load('../data/raw')
print(f'Sesiones totales cargadas : {len(df_raw):,}')
print(f'Columnas                  : {list(df_raw.columns)}')
print(f'Usuarios únicos           : {df_raw["userId"].nunique():,}')

In [ ]:
# Filtrar solo running + gender válido + heart_rate + GPS
df = filter_running_sessions(df_raw)
df = df[df['gender'].notna()]
df = df[df['heart_rate'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
df = df[df['latitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
df = df[df['altitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
print(f'Sesiones válidas para EDA : {len(df):,}')
print(f'Usuarios                  : {df["userId"].nunique():,}')

## 2. Extracción de estadísticos por sesión

In [ ]:
def extract_session_stats(row):
    """Extrae estadísticos escalares de las series temporales de una sesión."""
    hr = np.array(row['heart_rate'], dtype=float)
    hr = hr[~np.isnan(hr) & (hr > 0)]

    ts = np.array(row['timestamp'], dtype=float)
    if ts[-1] - ts[0] > 1e8:
        ts = ts / 1000.0
    duration_min = (ts[-1] - ts[0]) / 60.0

    alt = np.array(row['altitude'], dtype=float)
    alt = alt[~np.isnan(alt)]

    # Velocidad desde GPS (Haversine simplificado)
    lat = np.radians(np.array(row['latitude'], dtype=float))
    lon = np.radians(np.array(row['longitude'], dtype=float))
    dlat = np.diff(lat)
    dlon = np.diff(lon)
    a = np.sin(dlat/2)**2 + np.cos(lat[:-1])*np.cos(lat[1:])*np.sin(dlon/2)**2
    dist_km = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    dt_h = np.diff(ts) / 3600.0
    dt_h = np.where(dt_h <= 0, np.nan, dt_h)
    speed = dist_km / dt_h
    speed = speed[~np.isnan(speed) & (speed < 60)]

    return {
        'duration_min': duration_min,
        'hr_mean': float(np.mean(hr)) if len(hr) > 0 else np.nan,
        'hr_max': float(np.max(hr)) if len(hr) > 0 else np.nan,
        'hr_min': float(np.min(hr)) if len(hr) > 0 else np.nan,
        'speed_mean': float(np.mean(speed)) if len(speed) > 0 else np.nan,
        'speed_max': float(np.max(speed)) if len(speed) > 0 else np.nan,
        'altitude_mean': float(np.mean(alt)) if len(alt) > 0 else np.nan,
        'altitude_max': float(np.max(alt)) if len(alt) > 0 else np.nan,
    }

print('Extrayendo estadísticos de sesión (puede tardar ~2 min)...')
stats_df = pd.DataFrame([extract_session_stats(row) for _, row in df.iterrows()])
stats_df['userId'] = df['userId'].values
stats_df['gender'] = df['gender'].values
print(f'Shape: {stats_df.shape}')
stats_df.head(3)

## 2.1 Estadísticos descriptivos

In [ ]:
num_vars = ['duration_min', 'hr_mean', 'hr_max', 'hr_min', 'speed_mean', 'speed_max', 'altitude_mean']

desc = stats_df[num_vars].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]).T
desc.columns = ['count', 'mean', 'std', 'min', 'P25', 'P50', 'P75', 'P90', 'P95', 'max']
desc = desc.round(2)

print('Estadísticos descriptivos:')
print(desc.to_string())

desc.to_csv(REPORTS / 'descriptive_stats.csv')
print('\nGuardado: reports/eda/descriptive_stats.csv')

## 2.2 Histogramas de distribución

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

var_labels = {
    'duration_min': 'Duración (min)',
    'hr_mean': 'FC media (bpm)',
    'hr_max': 'FC máxima (bpm)',
    'hr_min': 'FC mínima (bpm)',
    'speed_mean': 'Velocidad media (km/h)',
    'speed_max': 'Velocidad máxima (km/h)',
    'altitude_mean': 'Altitud media (m)',
}

for i, (var, label) in enumerate(var_labels.items()):
    data = stats_df[var].dropna()
    axes[i].hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media={data.mean():.1f}')
    axes[i].axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label=f'Mediana={data.median():.1f}')
    axes[i].set_title(label, fontsize=11)
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='y', alpha=0.3)

axes[7].set_visible(False)  # espacio sobrante
plt.suptitle('Distribución de variables por sesión — FitRec', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_histogramas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eda_histogramas.png')

## 3. Detección de outliers (IQR + z-score)

In [ ]:
outlier_report = []

for var in num_vars:
    data = stats_df[var].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_iqr = Q1 - 1.5 * IQR
    upper_iqr = Q3 + 1.5 * IQR
    n_iqr = ((data < lower_iqr) | (data > upper_iqr)).sum()

    z_scores = np.abs(stats.zscore(data))
    n_zscore = (z_scores > 3).sum()

    outlier_report.append({
        'variable': var,
        'n_total': len(data),
        'outliers_IQR': n_iqr,
        'pct_IQR': round(100 * n_iqr / len(data), 2),
        'outliers_zscore': n_zscore,
        'pct_zscore': round(100 * n_zscore / len(data), 2),
        'lower_IQR': round(lower_iqr, 2),
        'upper_IQR': round(upper_iqr, 2),
    })

outlier_df = pd.DataFrame(outlier_report)
print(outlier_df.to_string(index=False))
outlier_df.to_csv(REPORTS / 'outlier_report.csv', index=False)
print('\nGuardado: reports/eda/outlier_report.csv')

In [ ]:
# Boxplots de las 4 variables principales
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

boxplot_vars = ['duration_min', 'hr_mean', 'speed_mean', 'altitude_mean']
boxplot_labels = ['Duración (min)', 'FC media (bpm)', 'Velocidad media (km/h)', 'Altitud media (m)']

for ax, var, label in zip(axes, boxplot_vars, boxplot_labels):
    data = stats_df[var].dropna()
    bp = ax.boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('Valor')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Boxplots — Detección de outliers por IQR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eda_boxplots.png')

## 4. Análisis de correlaciones

In [ ]:
corr_vars = ['duration_min', 'hr_mean', 'hr_max', 'hr_min', 'speed_mean', 'speed_max', 'altitude_mean']
corr_matrix = stats_df[corr_vars].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5, ax=ax,
    xticklabels=corr_vars, yticklabels=corr_vars
)
ax.set_title('Heatmap de correlaciones (Pearson) — variables por sesión', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

corr_matrix.to_csv(REPORTS / 'correlation_matrix.csv')
print('Guardado: reports/figures/eda_correlaciones.png')
print('Guardado: reports/eda/correlation_matrix.csv')

## 5. Sesgos demográficos

In [ ]:
# Distribución por sexo
gender_counts = df['gender'].value_counts()
print('Sesiones por sexo:')
print(gender_counts.to_string())
print(f'\nPorcentaje mujeres: {100*gender_counts.get("female", 0)/len(df):.1f}%')
print(f'Porcentaje hombres: {100*gender_counts.get("male", 0)/len(df):.1f}%')

In [ ]:
# Sesiones por usuario
sessions_per_user = df.groupby('userId').size()
print('Sesiones por usuario — estadísticos:')
print(sessions_per_user.describe([0.25, 0.5, 0.75, 0.90, 0.95]).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfico de sexo
gender_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'],
                   edgecolor='white', rot=0)
axes[0].set_title('Sesiones por sexo', fontsize=11)
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Nº sesiones')
for bar, count in zip(axes[0].patches, gender_counts):
    pct = 100 * count / gender_counts.sum()
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{pct:.1f}%', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Distribución sesiones por usuario
axes[1].hist(sessions_per_user, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(sessions_per_user.mean(), color='red', linestyle='--',
                label=f'Media={sessions_per_user.mean():.0f}')
axes[1].axvline(sessions_per_user.median(), color='orange', linestyle=':',
                label=f'Mediana={sessions_per_user.median():.0f}')
axes[1].set_title('Sesiones por usuario', fontsize=11)
axes[1].set_xlabel('Nº sesiones')
axes[1].set_ylabel('Nº usuarios')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Sesgos demográficos — FitRec', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_demografico.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eda_demografico.png')

In [ ]:
# FC media por sexo
print('FC media por sexo:')
print(stats_df.groupby('gender')['hr_mean'].describe().round(2).to_string())

print('\nVelocidad media por sexo:')
print(stats_df.groupby('gender')['speed_mean'].describe().round(2).to_string())

## 6. Análisis de span temporal por usuario

Identifica la **cohorte apta para ACWR canónico a 28 días** y cuantifica
el **sesgo de cobertura** (usuarios excluidos por span < 28 días).

In [ ]:
# Extraer fecha de inicio de cada sesión
def extract_session_date(ts_list):
    try:
        ts = min(ts_list)
        if ts > 1e9:
            ts /= 1000.0
        return pd.to_datetime(ts, unit='s').normalize()
    except Exception:
        return pd.NaT

df_span = df[['userId', 'timestamp']].copy()
df_span['date'] = df_span['timestamp'].apply(extract_session_date)
df_span = df_span[df_span['date'].notna()]

# Span por usuario
span_stats = df_span.groupby('userId')['date'].agg(
    primera_sesion='min',
    ultima_sesion='max',
    n_sesiones='count'
).reset_index()
span_stats['span_dias'] = (span_stats['ultima_sesion'] - span_stats['primera_sesion']).dt.days

print('Distribución del span temporal por usuario (días):')
print(span_stats['span_dias'].describe([0.25, 0.5, 0.75, 0.90]).round(1).to_string())

In [ ]:
# Cohorte apta para ACWR canónico (span >= 28 días)
ACWR_THRESHOLD = 28

n_total_users = len(span_stats)
n_eligible = (span_stats['span_dias'] >= ACWR_THRESHOLD).sum()
n_excluded = n_total_users - n_eligible

print(f'\n=== SESGO DE COBERTURA PARA ACWR CANÓNICO (umbral: {ACWR_THRESHOLD} días) ===')
print(f'Usuarios totales             : {n_total_users:,}')
print(f'Usuarios APTOS (span >= 28d) : {n_eligible:,}  ({100*n_eligible/n_total_users:.1f}%)')
print(f'Usuarios EXCLUIDOS (span < 28d): {n_excluded:,}  ({100*n_excluded/n_total_users:.1f}%)')

# Sesiones en la cohorte apta
eligible_users = span_stats[span_stats['span_dias'] >= ACWR_THRESHOLD]['userId']
n_sessions_eligible = df_span[df_span['userId'].isin(eligible_users)].shape[0]
n_sessions_total = len(df_span)
print(f'\nSesiones en cohorte apta     : {n_sessions_eligible:,}  ({100*n_sessions_eligible/n_sessions_total:.1f}%)')
print(f'Sesiones excluidas           : {n_sessions_total - n_sessions_eligible:,}')

# Guardar resumen
cohort_summary = pd.DataFrame([{
    'umbral_dias': ACWR_THRESHOLD,
    'total_usuarios': n_total_users,
    'usuarios_aptos': n_eligible,
    'pct_aptos': round(100*n_eligible/n_total_users, 2),
    'usuarios_excluidos': n_excluded,
    'pct_excluidos': round(100*n_excluded/n_total_users, 2),
    'sesiones_cohorte_apta': n_sessions_eligible,
    'pct_sesiones_aptas': round(100*n_sessions_eligible/n_sessions_total, 2),
}])
cohort_summary.to_csv(REPORTS / 'acwr_cohort_coverage.csv', index=False)
print('\nGuardado: reports/eda/acwr_cohort_coverage.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma del span temporal
axes[0].hist(span_stats['span_dias'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(ACWR_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Umbral ACWR ({ACWR_THRESHOLD} días)')
axes[0].axvline(span_stats['span_dias'].median(), color='orange', linestyle=':',
                linewidth=1.5, label=f"Mediana={span_stats['span_dias'].median():.0f}d")
axes[0].set_title('Distribución del span temporal por usuario', fontsize=11)
axes[0].set_xlabel('Span (días)')
axes[0].set_ylabel('Nº usuarios')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de barras sesgo de cobertura
coverage_data = [n_eligible, n_excluded]
coverage_labels = [f'Aptos\n(span≥28d)\n{n_eligible} ({100*n_eligible/n_total_users:.0f}%)',
                   f'Excluidos\n(span<28d)\n{n_excluded} ({100*n_excluded/n_total_users:.0f}%)']
bars = axes[1].bar(coverage_labels, coverage_data, color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_title('Sesgo de cobertura — Cohorte ACWR', fontsize=11)
axes[1].set_ylabel('Nº usuarios')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Análisis de span temporal — Cohorte apta para ACWR canónico', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_span_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eda_span_temporal.png')

In [ ]:
# Relación span vs número de sesiones
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(span_stats['span_dias'], span_stats['n_sesiones'],
                alpha=0.3, s=10, c='steelblue')
ax.axvline(ACWR_THRESHOLD, color='red', linestyle='--', label=f'Umbral {ACWR_THRESHOLD}d')
ax.set_xlabel('Span temporal (días)')
ax.set_ylabel('Número de sesiones')
ax.set_title('Span temporal vs. nº sesiones por usuario', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_span_vs_sesiones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eda_span_vs_sesiones.png')

In [ ]:
span_stats.to_csv(REPORTS / 'span_por_usuario.csv', index=False)
print('Guardado: reports/eda/span_por_usuario.csv')
print('\n=== EDA completado ===')
print(f'Outputs en reports/eda/ y reports/figures/')